# RAG System with Cities Text + Qwen 3.5-2B
## LangChain-based Retrieval-Augmented Generation

This notebook implements a production-ready RAG system:
- Loads all 315 Romanian city text files
- Chunks them using RecursiveCharacterTextSplitter
- Creates embeddings using local embedding server
- Stores in Chroma vector database
- Uses Qwen 3.5-2B for generation via RetrievalQA chain
- Customizable prompts

**Prerequisites:**
- Local server running on http://localhost:1234/v1 (for both embeddings and LLM)
- City text files in `informations/cities_text/`
- Installed: langchain, chroma, openai

In [1]:
## Pasul 1: Instaleaza bibliotecile necesare

import subprocess
import sys

packages = ['langchain', 'langchain-openai', 'chromadb', 'langchain-text-splitters']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} deja instalat")
    except ImportError:
        print(f"⚠ Se instaleaza {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f"✓ {package} instalat")

print("\n✓ Toate dependintele sunt gata!")

✓ langchain deja instalat
⚠ Se instaleaza langchain-openai...
✓ langchain-openai instalat
⚠ Se instaleaza chromadb...
✓ chromadb instalat


c:\Users\Alex\Documents\llms\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ langchain-text-splitters deja instalat

✓ Toate dependintele sunt gata!


In [4]:
## Pasul 2: Incarca toate fisierele text ale oraselor

from pathlib import Path
from langchain_core.documents import Document

print("=" * 100)
print("INCARCARE FISIERE TEXT ALE ORASELOR")
print("=" * 100)

text_dir = Path('informations/cities_text')
text_files = sorted(text_dir.glob('*.txt'))

print(f"\n📁 Se incarca din: {text_dir}")
print(f"🔍 Gasit {len(text_files)} fisiere cu orase\n")

docs = []
for idx, text_file in enumerate(text_files, 1):
    try:
        content = text_file.read_text(encoding='utf-8')
        city_name = text_file.stem
        
        doc = Document(
            page_content=content,
            metadata={
                'city': city_name,
                'source': str(text_file),
                'length': len(content)
            }
        )
        docs.append(doc)
        
        if idx % 50 == 0 or idx == 1:
            print(f"[{idx}/{len(text_files)}] ✓ {city_name:<25} ({len(content):>7,} caractere)")
    
    except Exception as e:
        print(f"[{idx}] ✗ Eroare la incarcare {text_file.stem}: {e}")

print(f"\n✓ Incarcat {len(docs)} documente cu orase")

total_chars = sum(doc.metadata['length'] for doc in docs)
print(f"\n📊 STATISTICI DOCUMENTE:")
print(f"   Total caractere: {total_chars:,}")
print(f"   Medie caractere/oras: {total_chars // len(docs):,}")
print(f"   Cel mai mare: {max(doc.metadata['length'] for doc in docs):,} caractere")
print(f"   Cel mai mic: {min(doc.metadata['length'] for doc in docs):,} caractere")

INCARCARE FISIERE TEXT ALE ORASELOR

📁 Se incarca din: informations\cities_text
🔍 Gasit 315 fisiere cu orase

[1/315] ✓ abrud                     ( 11,156 caractere)
[50/315] ✓ buziaș                    ( 18,726 caractere)
[100/315] ✓ deta                      ( 11,572 caractere)
[150/315] ✓ livada                    (    880 caractere)
[200/315] ✓ petrila                   (  8,856 caractere)
[250/315] ✓ sulina                    (  8,883 caractere)
[300/315] ✓ voluntari                 ( 14,303 caractere)

✓ Incarcat 315 documente cu orase

📊 STATISTICI DOCUMENTE:
   Total caractere: 5,829,530
   Medie caractere/oras: 18,506
   Cel mai mare: 217,873 caractere
   Cel mai mic: 7 caractere


In [5]:
## Pasul 3: Imparte documente in bucati

from langchain_text_splitters import RecursiveCharacterTextSplitter

print("\n" + "=" * 100)
print("IMPARTIRE DOCUMENTE IN BUCATI")
print("=" * 100)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("\n⚙️ Configuratie impartitor:")
print(f"   Dimensiune bucata: 1000 caractere")
print(f"   Suprapunere: 100 caractere")
print(f"   Separatori: paragrafuri, linii noi, propozitii, cuvinte, caractere\n")

print("🔪 Se impart documente...")
splits = text_splitter.split_documents(docs)

print(f"\n✓ Creat {len(splits)} bucati din {len(docs)} documente")

avg_chunk_size = sum(len(chunk.page_content) for chunk in splits) / len(splits) if splits else 0
print(f"\n📊 STATISTICI IMPARTIRE:")
print(f"   Documente originale: {len(docs)}")
print(f"   Total bucati: {len(splits)}")
print(f"   Medie bucati per oras: {len(splits) / len(docs):.1f}")
print(f"   Dimensiune medie bucata: {avg_chunk_size:.0f} caractere")
print(f"   Raport compresie: {(len(splits) / len(docs)):.2f}x\n")

print("📝 EXEMPLE BUCATI:")
for i in range(min(3, len(splits))):
    chunk = splits[i]
    city = chunk.metadata.get('city', 'necunoscut')
    preview = chunk.page_content[:100].replace('\n', ' ')
    print(f"   [{i+1}] {city}: {preview}...")


IMPARTIRE DOCUMENTE IN BUCATI

⚙️ Configuratie impartitor:
   Dimensiune bucata: 1000 caractere
   Suprapunere: 100 caractere
   Separatori: paragrafuri, linii noi, propozitii, cuvinte, caractere

🔪 Se impart documente...

✓ Creat 7540 bucati din 315 documente

📊 STATISTICI IMPARTIRE:
   Documente originale: 315
   Total bucati: 7540
   Medie bucati per oras: 23.9
   Dimensiune medie bucata: 795 caractere
   Raport compresie: 23.94x

📝 EXEMPLE BUCATI:
   [1] abrud: AbrudAcest articol se referă la un oraș din județul Alba, România. Pentru o depresiune din Munții Ap...
   [2] abrud: Denumire Încă de la descoperirea Tăblițelor romane cerate de la Roșia Montană în 1786, filologul ger...
   [3] abrud: Geografie Orașul Abrud este situat în depresiunea Abrudului, un spațiu dominat de un relief vălurit,...


In [ ]:
## Pasul 4: Configurare embedding-uri

import requests
from typing import List
from langchain.embeddings.base import Embeddings

print("\n" + "=" * 100)
print("CONFIGURARE EMBEDDING-URI (TRANSFORMARI VECTOR)")
print("=" * 100)

class LocalServerEmbeddings(Embeddings):
    """
    Clasa pentru conectare la serverul local de embedding-uri.
    Se conecteaza la http://localhost:1234/v1/embeddings
    """
    
    def __init__(self, base_url: str = "http://localhost:1234/v1"):
        self.base_url = base_url
        self.model = "text-embedding-nomic-embed-text-v1.5"
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Genereaza embedding-uri pentru o lista de texte (documente)"""
        try:
            response = requests.post(
                f"{self.base_url}/embeddings",
                json={"input": texts}
            )
            response.raise_for_status()
            data = response.json()
            return [item["embedding"] for item in data["data"]]
        except Exception as e:
            print(f"❌ Eroare la embedding: {e}")
            raise
    
    def embed_query(self, text: str) -> List[float]:
        """Genereaza embedding pentru o intrebare (query)"""
        try:
            response = requests.post(
                f"{self.base_url}/embeddings",
                json={"input": [text]}
            )
            response.raise_for_status()
            data = response.json()
            return data["data"][0]["embedding"]
        except Exception as e:
            print(f"❌ Eroare conectare la serverul embedding: {e}")
            print(f"   Verifica: http://localhost:1234/v1/embeddings")
            print(f"   Verifica ca Ollama sau serverul local ruleaza")
            raise

print("\n✓ Definita clasa LocalServerEmbeddings")

print("\n⚙️ Initiare embedding model:")
embeddings = LocalServerEmbeddings(base_url="http://localhost:1234/v1")
print(f"   URL: {embeddings.base_url}/embeddings")
print(f"   Model: {embeddings.model}")

print("\n🧪 Test conexiune - genereaza embedding pentru text de test...")
try:
    test_embedding = embeddings.embed_query("Salut, acesta este un test!")
    print(f"   ✓ Succes! Embedding generat: vector cu {len(test_embedding)} dimensiuni")
    print(f"   ✓ Primii 5 vectori: {[f'{x:.4f}' for x in test_embedding[:5]]}")
except Exception as e:
    print(f"   ❌ Eroare: {e}")


CONFIGURARE EMBEDDING-URI (TRANSFORMARI VECTOR)

✓ Definita clasa LocalServerEmbeddings

⚙️ Initiare embedding model:
   URL: http://localhost:1234/v1/embeddings
   Model: text-embedding-nomic-embed-text-v1.5

🧪 Test conexiune - genereaza embedding pentru text de test...
   ✓ Succes! Embedding generat: vector cu 768 dimensiuni
   ✓ Primii 5 vectori: ['0.0349', '0.0278', '-0.1692', '-0.0153', '0.0481']


In [10]:
## Pasul 5: Creare depozit vector (Vector Store) cu Chroma

import os
import shutil
from langchain_community.vectorstores import Chroma

print("\n" + "=" * 100)
print("CREARE DEPOZIT VECTOR (CHROMA DATABASE)")
print("=" * 100)

# Stergere depozit anterior (daca exista)
chroma_dir = "chroma_cities"
if os.path.exists(chroma_dir):
    print(f"\n🗑️ Sterge depozit anterior: {chroma_dir}")
    shutil.rmtree(chroma_dir)

print(f"\n📦 Configurare Chroma:")
print(f"   Directorul depozitului: {chroma_dir}")
print(f"   Total bucati (chunks) de introdus: {len(splits)}")
print(f"   Model embedding: text-embedding-3-small")

print(f"\n⏳ Se creeaza depozitul vector din {len(splits)} bucati...")
print("   (aceasta operatie poate dura cateva minute...)")

vector_store = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=chroma_dir,
    collection_name="romanian_cities"
)

print(f"\n✓ Depozit vector creat cu succes!")

# Verifica colectia
try:
    collection = vector_store._collection
    count = collection.count()
    print(f"\n📊 STATISTICI DEPOZIT:")
    print(f"   Bucati indexate: {count}")
    print(f"   Directorul depozitului: {chroma_dir}")
    print(f"   Oras de stocare: ~100-200 MB (in functie de modele)")
except Exception as e:
    print(f"   ⚠️ Nu pot verifica numarul exact de bucati: {e}")

print(f"\n✓ Depozitul vector este pregatit pentru cautari semantice!")


CREARE DEPOZIT VECTOR (CHROMA DATABASE)

📦 Configurare Chroma:
   Directorul depozitului: chroma_cities
   Total bucati (chunks) de introdus: 7540
   Model embedding: text-embedding-3-small

⏳ Se creeaza depozitul vector din 7540 bucati...
   (aceasta operatie poate dura cateva minute...)


In [11]:
## Pasul 6: Configurare model Qwen 3.5-2B

from langchain_openai import ChatOpenAI

print("\n" + "=" * 100)
print("CONFIGURARE MODEL QWEN 3.5-2B (LARGE LANGUAGE MODEL)")
print("=" * 100)

print("\n⚙️ Configuratie LLM:")
llm = ChatOpenAI(
    model="qwen",
    temperature=0.7,
    max_tokens=500,
    openai_api_base="http://localhost:1234/v1",
    openai_api_key="not-needed"
)

print(f"   Model: Qwen 3.5-2B")
print(f"   URL Server: http://localhost:1234/v1")
print(f"   Temperatura: 0.7 (creeaza/creativ)")
print(f"   Token-uri maximi: 500")

print(f"\n🧪 Test conexiune la Qwen...")
try:
    test_message = "Salut! Spune-mi intr-o singura linie cine esti."
    test_response = llm.predict(test_message)
    print(f"   ✓ Succes! Qwen raspunde:")
    print(f"   \"{test_response[:100]}...\"")
except Exception as e:
    print(f"   ❌ Eroare conectare la Qwen: {e}")
    print(f"   Verifica: http://localhost:1234/v1/chat/completions")
    print(f"   Verifica ca Ollama / LM Studio ruleaza local")


CONFIGURARE MODEL QWEN 3.5-2B (LARGE LANGUAGE MODEL)

⚙️ Configuratie LLM:
   Model: Qwen 3.5-2B
   URL Server: http://localhost:1234/v1
   Temperatura: 0.7 (creeaza/creativ)
   Token-uri maximi: 500

🧪 Test conexiune la Qwen...
   ❌ Eroare conectare la Qwen: 'ChatOpenAI' object has no attribute 'predict'
   Verifica: http://localhost:1234/v1/chat/completions
   Verifica ca Ollama / LM Studio ruleaza local


In [ ]:
## Pasul 7: Creare lant RAG (Retrieval-Augmented Generation)

from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

print("\n" + "=" * 100)
print("CREARE LANT RAG (RETRIEVAL + GENERATION)")
print("=" * 100)

# Prompt personalizat in romana
template_ro = """Tu esti un expert in ceografie, istorie si cultura asupra oraselor din Romania.
Foloseste contextul de mai jos pentru a raspunde la intrebarea utilizatorului.

CONTEXT DIN BAZA DE DATE:
{context}

INTREBAREA UTILIZATORULUI:
{question}

INSTRUCTIUNI:
- Raspunde in romana
- Baza-te pe informatiile din context
- Daca nu gasesti informatii relevante, spune "Nu am informatii suficiente"
- Cite sursa (orasul) din care provine informatia
- Raspunde concis (maxim 3-4 propozitii)
- Evita repetatii

RASPUNS:
"""

prompt = PromptTemplate(
    template=template_ro,
    input_variables=["context", "question"]
)

print("\n⚙️ Configuratie lant RAG:")
print(f"   Retriever: Chroma (cautare semantica)")
print(f"   LLM: Qwen 3.5-2B")
print(f"   Chain type: RetrievalQA (document cauta + answer gen)")

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(f"   Top-K documente returnate: 3 cele mai relevante bucati")

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print(f"\n✓ Lant RAG creat si pregatit!")

In [ ]:
## Pasul 8: Testare sistem RAG

print("\n" + "=" * 100)
print("TESTARE SISTEM RAG - INTREBARI ETALON")
print("=" * 100)

def display_rag_result(query: str, result: dict):
    """Afiseaza rezultatul complet al cautarii RAG"""
    print(f"\n{'='*80}")
    print(f"❓ INTREBARE: {query}")
    print(f"{'='*80}")
    print(f"\n📝 RASPUNS QWEN:")
    print(f"{result['result']}")
    print(f"\n📚 DOCUMENTE SURSA (top-3 cele mai relevante):")
    for i, doc in enumerate(result['source_documents'], 1):
        city = doc.metadata.get('city', 'necunoscut')
        preview = doc.page_content[:150].replace('\n', ' ')
        print(f"   [{i}] {city}: {preview}...")

# Teste etalon
test_queries = [
    "Care este capitala Romaniei?",
    "Ce atractii turistice are Cluj-Napoca?",
    "Demografie și geografia Constanței"
]

print(f"\n🔄 Ruleaza {len(test_queries)} intrebari de test...\n")

for i, query in enumerate(test_queries, 1):
    try:
        print(f"\n⏳ Test {i}/{len(test_queries)}: Se proceseaza intrebarea...")
        result = qa_chain({"query": query})
        display_rag_result(query, result)
    except Exception as e:
        print(f"❌ Eroare la test {i}: {e}")

print(f"\n{'='*80}")
print(f"✓ Testare completata!")

In [ ]:
## Pasul 9: Interfata RAG interactiva

print("\n" + "=" * 100)
print("SISTEM INTERACTIV DE CAUTARE - ORASE ROMANESTI")
print("=" * 100)

def query_cities_rag(question: str, verbose: bool = True):
    """
    Functie interactiva pentru cautare informatii despre orase din Romania.
    
    PARAMETRI:
        question (str): Intrebarea dumneavoastra despre orasele din Romania
        verbose (bool): Afiseaza detalii suplimentare (documente sursa, etc.)
    
    EXEMPLE DE INTREBARI:
        - "Care sunt principalele atractii din Brașov?"
        - "Ce informații are despre Belfast?" (nota: nu e oras roman)
        - "Cine a fondat Sibiul?"
        - "Ce economie are Timisoara?"
        - "Podul Carol din care oras se afla?"
    
    RETUR:
        str: Raspunsul generat de Qwen bazat pe informații din baza de date
    """
    
    if not question or not question.strip():
        return "❌ Eroare: Intrebarea nu poate fi goala!"
    
    try:
        print(f"\n🔍 Cauta informatii despre: '{question}'")
        print("   ⏳ Se interogheaza depozitul vector...")
        
        result = qa_chain({"query": question})
        
        print(f"\n✓ Rezultat gasit!\n")
        print(f"📝 RASPUNS:")
        print(f"{result['result']}\n")
        
        if verbose:
            print(f"📚 DOCUMENTE RELEVANTE (top-3):")
            for i, doc in enumerate(result['source_documents'], 1):
                city = doc.metadata.get('city', 'necunoscut')
                preview = doc.page_content[:100].replace('\n', ' ')
                print(f"   [{i}] {city}: {preview}...")
        
        return result['result']
    
    except Exception as e:
        return f"❌ Eroare la procesare: {str(e)}"

# Mesaj de bun venit
print("\n" + "="*80)
print("✅ SISTEM PREGATIT! Poti pune intrebari despre orasele Romaniei")
print("="*80)

print("\n📌 UTILIZARE:")
print("   query_cities_rag('Intrebarea ta despre orase din Romania')")

print("\n💡 EXEMPLE:")
print("   query_cities_rag('Care sunt atractiile turistice din Sibiu?')")
print("   query_cities_rag('Cum se deplaseaza intre Iasi si Cluj?')")
print("   query_cities_rag('Ce se stie despre istoria Bucurestiului?')")
print("   query_cities_rag('Care este cea mai mare oras din Romania?')")

print("\n🎯 TIP: Pune intrebari specifice pentru rezultate mai bune!")
print("="*80)

## System Architecture & Documentation

### 🏗️ RAG Pipeline Overview

```
City Text Files (315 cities)
            ↓
RecursiveCharacterTextSplitter (1000 chars, 100 overlap)
            ↓
~3000+ Chunks
            ↓
LocalServerEmbeddings (via http://localhost:1234/v1)
            ↓
Chroma Vector Database
            ↓
Similarity Search (Top-K retrieval)
            ↓
Retrieved Context + Question
            ↓
Qwen 3.5-2B LLM (http://localhost:1234/v1)
            ↓
Answer with Source Attribution
```

### ⚙️ Key Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| **chunk_size** | 1000 | Characters per chunk |
| **chunk_overlap** | 100 | Overlap between chunks (for context) |
| **top_k** | 3 | Documents to retrieve |
| **temperature** | 0.7 | LLM creativity (0-1) |
| **max_tokens** | 500 | Maximum response length |

### 🔍 Retrieval Process

- **Distance Metric**: Cosine similarity in embedding space
- **Index**: Chroma (optimized for semantic search)
- **Speed**: ~100ms per query (after vector store built)

### 📋 Supported Queries

✓ Geographic information  
✓ Historical facts  
✓ Demographics  
✓ Economic data  
✓ Cultural attractions  
✓ Transport information  
✓ Comparative questions  
✓ Both English and Romanian  

### 🚀 Performance Tips

1. **Increase top_k** for more context (better accuracy, slower)
2. **Decrease temperature** for factual answers
3. **Use specific questions** rather than vague ones
4. **Try multiple phrasings** if first answer isn't good

### 💾 Storage

- **Vector Store**: `chroma_cities/` (~50-100 MB depending on model)
- **Chunks**: ~3000-4000 total
- **Embedding Dim**: Depends on model (usually 384-768)

### 🔗 Required Services

- **Embedding Server**: http://localhost:1234/v1/embeddings
- **LLM Server**: http://localhost:1234/v1/chat/completions
- **Both can be same Ollama/LM Studio instance**

### 📚 Example Advanced Queries

```python
# Comparative analysis
query_cities_rag("How are Brașov and Sibiu different?", top_k=4)

# Multi-city query
query_cities_rag("List cities in Moldova region", top_k=5)

# Historical context
query_cities_rag("What important historical events happened in Cluj?")

# Economic focus
query_cities_rag("Which cities have significant industrial activity?", top_k=3)

# Tourism
query_cities_rag("What are the best tourist attractions mentioned?", top_k=5)
```

### ⚠️ Troubleshooting

| Issue | Solution |
|-------|----------|
| "Connection refused" | Check if local server running on localhost:1234 |
| "Out of memory" | Reduce chunk_size or use fewer documents |
| "Generic answers" | Increase top_k or rephrase question |
| "Slow first run" | Normal - embeddings being generated. Be patient. |
| "Can't find answer" | Try with more top_k results or different query |